# Main Cloud Orchestrator - Colab & Kaggle

This notebook serves as the main orchestrator for running experiments on Google Colab or Kaggle. It performs a sparse checkout to download only the lightweight code directory (`experiments/`), installs the package in editable mode, and executes python modules in the notebook space to preserve memory and variables for debugging.

In [ ]:
# Cell 1: Sparse Repository Synchronization and Cloud Package Installation
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'  # Replace with actual repository URL if necessary

# 1. Perform sparse checkout of only the 'experiments/' directory to avoid downloading heavy PDFs
if not os.path.exists(REPO_NAME):
    print(f"Cloning repository {REPO_NAME} (sparse checkout for 'experiments/')...")
    !git clone -q --filter=blob:none --no-checkout {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    !git checkout -q
    %cd experiments
else:
    print(f"Updating repository {REPO_NAME}...")
    %cd {REPO_NAME}
    !git pull -q
    %cd experiments

# 2. Verify working directory is 'experiments'
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Failed to navigate to experiments directory. Current path: {current_dir}")

# 3. Install the package in editable mode with the [cloud] extra
print("Installing experiments package with [cloud] dependencies...")
%pip install -q -e .[cloud]

In [ ]:
# Cell 2: Mount Google Drive to load/save datasets
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Run colocated unit tests with pytest
!pytest src/

In [ ]:
# Cell 4: Run the production parser on the full train.csv dataset stored in Google Drive
# Default paths are built-in, no arguments needed!
%run src/data_preparation/parser.py

In [ ]:
# Cell 5: Terminate process and automatically disconnect Colab VM to save credits
print("Proceso terminado.")
from google.colab import runtime
runtime.unassign()